In [0]:
dbutils.widgets.text("p_data_source", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
%run "../Includes/configs" 

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/circuits.csv,circuits.csv,10044,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/constructors.json,constructors.json,30415,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/drivers.json,drivers.json,180812,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/lap_times/,lap_times/,0,1767877145000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/pit_stops.json,pit_stops.json,1369387,1767877119000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/qualifying/,qualifying/,0,1767877183000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/races.csv,races.csv,116847,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/results.json,results.json,7165641,1767877120000


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
circuitId,circuitRef,name,location,country,lat,lng,alt,url
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_International_Circuit
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,http://en.wikipedia.org/wiki/Bahrain_International_Circuit
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,http://en.wikipedia.org/wiki/Istanbul_Park
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers_Magny-Cours
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
qualifying_schema = StructType(fields=[StructField("qualifyId", IntegerType(), False),
                                       StructField("raceId", IntegerType(), True),
                                       StructField("driverId", IntegerType(), True),
                                       StructField("constructorId", IntegerType(), True),
                                       StructField("number", IntegerType(), True),
                                       StructField("position", IntegerType(), True),
                                       StructField("q1", StringType(), True),
                                       StructField("q2", StringType(), True),
                                       StructField("q3", StringType(), True)])

In [0]:
qualifying_df = spark.read.option("multiline", True).schema(qualifying_schema).json(f"{raw_folder_path}/qualifying")

In [0]:
display(qualifying_df)

qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
5,18,2,2,3,5,1:25.960,1:25.518,1:27.236
6,18,15,7,11,6,1:26.427,1:26.101,1:28.527
7,18,3,3,7,7,1:26.295,1:26.059,1:28.687
8,18,14,9,9,8,1:26.381,1:26.063,1:29.041
9,18,10,7,12,9,1:26.919,1:26.164,1:29.593
10,18,20,5,15,10,1:26.702,1:25.842,\N


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
qualifying_final_df = qualifying_df.withColumnRenamed("qualifyId", "qualify_id").withColumnRenamed("raceId", "race_id").withColumnRenamed("driverId", "driver_id").withColumnRenamed("constructorId", "constructor_id").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source))

In [0]:
display(qualifying_final_df)

qualify_id,race_id,driver_id,constructor_id,number,position,q1,q2,q3,ingestion_date,data_source
1,18,1,1,22,1,1:26.572,1:25.187,1:26.714,2026-01-10T11:39:54.357411Z,Ergast API
2,18,9,2,4,2,1:26.103,1:25.315,1:26.869,2026-01-10T11:39:54.357411Z,Ergast API
3,18,5,1,23,3,1:25.664,1:25.452,1:27.079,2026-01-10T11:39:54.357411Z,Ergast API
4,18,13,6,2,4,1:25.994,1:25.691,1:27.178,2026-01-10T11:39:54.357411Z,Ergast API
5,18,2,2,3,5,1:25.960,1:25.518,1:27.236,2026-01-10T11:39:54.357411Z,Ergast API
6,18,15,7,11,6,1:26.427,1:26.101,1:28.527,2026-01-10T11:39:54.357411Z,Ergast API
7,18,3,3,7,7,1:26.295,1:26.059,1:28.687,2026-01-10T11:39:54.357411Z,Ergast API
8,18,14,9,9,8,1:26.381,1:26.063,1:29.041,2026-01-10T11:39:54.357411Z,Ergast API
9,18,10,7,12,9,1:26.919,1:26.164,1:29.593,2026-01-10T11:39:54.357411Z,Ergast API
10,18,20,5,15,10,1:26.702,1:25.842,\N,2026-01-10T11:39:54.357411Z,Ergast API


In [0]:
qualifying_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/qualifying")